# SageMaker AI MLflow — Dataset Curation from Production Traces

In this notebook, you will curate a high-quality dataset from production traces captured across multiple MLflow experiments — both agent autolog traces and DataCapture pipeline traces. This curated dataset will be used in the next notebook to fine-tune a smaller model.

**What you will learn:**
- Load and filter MLflow traces from the online evaluation experiment using evaluation scores
- Prepare an SFT dataset in `prompt`/`completion` format from curated traces
- Register the dataset in the SageMaker AI Registry using `DataSet.create()`

### Prerequisites
- Completed the agent tracing lab (04-1) with traces in `medical-triage-agent`
- Completed the evaluation labs with traces in `medical-triage-datacapture` and `medical-triage-offline-eval`
- A SageMaker AI MLflow App


## 1. Environment Setup

This module uses the **SageMaker Python SDK v3** (`sagemaker>=3.5.0`) which provides the `SFTTrainer`, `DataSet` registry, and new resource APIs.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
# Install SageMaker SDK v3 and dependencies
%pip install "sagemaker>=3.5.0" mlflow==3.9.0 sagemaker-mlflow==0.2.0 datasets==4.5.0 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

In [ ]:
import json
import os
import boto3
import mlflow
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name
s3_client = boto3.client("s3")

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

print(f"Role: {role}")
print(f"Bucket: {bucket_name}")
print(f"Region: {region}")

## 2. Configure SageMaker AI MLflow

Connect to your SageMaker AI MLflow App. We will read traces from the online evaluation experiment and create a new experiment for the fine-tuning run.

In [ ]:
# Retrieve values stored from previous labs
%store -r

%store
if MLFLOW_TRACKING_URI is None or MLFLOW_TRACKING_URI == '':
    print("MLFLOW_TRACKING_URI not found. Please enter your MLflow App ARN.")
MLFLOW_TRACKING_URI

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Source experiments (traces from agent, DataCapture pipeline, and offline evaluation)
agent_experiment_name = "medical-triage-agent"
datacapture_experiment_name = "medical-triage-datacapture"
offline_eval_experiment_name = "medical-triage-offline-eval"

# New experiment for fine-tuning tracking
ft_experiment_name = "medical-triage-fine-tuning"
mlflow.set_experiment(ft_experiment_name)

print(f"MLflow tracking server: {mlflow.get_tracking_uri()}")
print(f"Source experiments: {agent_experiment_name}, {datacapture_experiment_name}, {offline_eval_experiment_name}")
print(f"Fine-tuning experiment: {ft_experiment_name}")

## 3. Load and Filter Traces from MLflow

We load all traces from the online evaluation experiment. Each trace contains:
- **Inputs**: The user's medical triage query
- **Outputs**: The Qwen3-8B model's response
- **Assessments**: Evaluation scores from the online evaluation scorers (Safety, RelevanceToQuery, etc.)

We filter to keep only high-quality traces where the model produced safe, relevant responses. These curated traces become the training data for the smaller model.

In [ ]:
# Load traces from the agent experiment
mlflow.set_experiment(agent_experiment_name)
agent_traces = mlflow.search_traces()
print(f"Traces in '{agent_experiment_name}': {len(agent_traces)}")

# Load traces from the DataCapture experiment
mlflow.set_experiment(datacapture_experiment_name)
datacapture_traces = mlflow.search_traces()
print(f"Traces in '{datacapture_experiment_name}': {len(datacapture_traces)}")

# Load traces from the offline evaluation experiment
mlflow.set_experiment(offline_eval_experiment_name)
offline_eval_traces = mlflow.search_traces()
print(f"Traces in '{offline_eval_experiment_name}': {len(offline_eval_traces)}")

# Combine traces from all experiments
all_traces = pd.concat([agent_traces, datacapture_traces, offline_eval_traces], ignore_index=True)
print(f"\nTotal combined traces: {len(all_traces)}")

# Switch back to fine-tuning experiment for logging
mlflow.set_experiment(ft_experiment_name)
all_traces.head()

### Inspect Available Columns

Let's see what columns are available in the traces DataFrame to understand the evaluation results structure.

In [ ]:
print("Available columns:")
for col in all_traces.columns:
    print(f"  {col}")

### Filter High-Quality Traces

We filter traces based on evaluation scores. Currently, we check for **safety** and **relevance** — only traces that passed these checks are included in the training dataset.

> **Tip:** To further refine the dataset, you can extend the filter logic below to exclude traces with negative human feedback (thumbs down) or poor scores on other metrics like coherence, professional tone, or bias. This is where you would add those additional filter conditions.

> **Note:** The exact column names for assessment results depend on how MLflow stores them. Adjust the filter conditions below based on the columns you see above. If assessment columns are not directly available, we will use all traces with valid inputs and outputs.

In [ ]:
# Extract prompt and completion from trace inputs/outputs
# Filter: only include traces that passed safety and relevance checks
curated_records = []
skipped_quality = 0

def check_assessment(assessments, name):
    """Check if a specific assessment passed (value='yes'). Returns True if passed or not found."""
    if not assessments:
        return True  # No assessments = include by default
    for a in assessments:
        if a.get('assessment_name') == name:
            feedback = a.get('feedback', {})
            return str(feedback.get('value', 'yes')).lower() != 'no'
    return True  # Assessment not found = include by default

for idx, trace in all_traces.iterrows():
    try:
        # Check safety and relevance assessments
        assessments = trace.get('assessments', [])
        if not check_assessment(assessments, 'safety') or not check_assessment(assessments, 'relevance_to_query'):
            skipped_quality += 1
            continue

        # Extract input (user query)
        inputs = trace.get("request")
        outputs = trace.get("response")
        
        if inputs is None or outputs is None:
            continue
        
        # Parse inputs — may be a string or dict
        if isinstance(inputs, str):
            try:
                inputs = json.loads(inputs)
            except json.JSONDecodeError:
                pass
        
        # Parse outputs — may be a string or dict
        if isinstance(outputs, str):
            try:
                outputs = json.loads(outputs)
            except json.JSONDecodeError:
                pass
        
        # Extract the prompt text
        if isinstance(inputs, dict):
            prompt = inputs.get("prompt", str(inputs))
        else:
            prompt = str(inputs)
        
        # Extract the completion text
        if isinstance(outputs, dict):
            completion = outputs.get("response", str(outputs))
        else:
            completion = str(outputs)
        
        # Skip empty or very short responses
        if len(completion.strip()) < 20:
            continue
        
        # Remove /no_think suffix from prompts if present
        prompt = prompt.replace(" /no_think", "").strip()
        
        curated_records.append({
            "prompt": prompt,
            "completion": completion,
        })
    except Exception as e:
        print(f"Skipping trace {idx}: {e}")
        continue

print(f"Curated {len(curated_records)} records from {len(all_traces)} traces")
print(f"Skipped {skipped_quality} traces that failed safety or relevance checks")

In [ ]:
# Preview a sample record
if curated_records:
    sample = curated_records[0]
    print(f"Sample prompt:\n{sample['prompt'][:300]}")
    print(f"\nSample completion:\n{sample['completion'][:300]}")

In [ ]:
all_records = curated_records
print(f"Total dataset size: {len(all_records)} records from production traces")

## 4. Prepare SFT Dataset

Convert the curated records into the SFT format expected by the SageMaker `SFTTrainer`. The trainer expects JSONL files with `prompt` and `completion` fields.

We split the data into training (80%) and validation (20%) sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train and validation
train_records, val_records = train_test_split(all_records, test_size=0.2, random_state=42)

print(f"Training samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")

In [ ]:
# Save as JSONL files locally
os.makedirs("./sft_data/train", exist_ok=True)
os.makedirs("./sft_data/val", exist_ok=True)

def save_jsonl(records, filepath):
    with open(filepath, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

save_jsonl(train_records, "./sft_data/train/dataset.jsonl")
save_jsonl(val_records, "./sft_data/val/dataset.jsonl")

print("Saved JSONL files:")
print(f"  Train: ./sft_data/train/dataset.jsonl ({len(train_records)} records)")
print(f"  Val:   ./sft_data/val/dataset.jsonl ({len(val_records)} records)")

In [ ]:
# Upload to S3
if default_prefix:
    s3_dataset_prefix = f"{default_prefix}/datasets/medical-triage-sft"
else:
    s3_dataset_prefix = "datasets/medical-triage-sft"

train_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/train/dataset.jsonl"
val_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/val/dataset.jsonl"

s3_client.upload_file("./sft_data/train/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/train/dataset.jsonl")
s3_client.upload_file("./sft_data/val/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/val/dataset.jsonl")

print(f"Uploaded to S3:")
print(f"  Train: {train_s3_path}")
print(f"  Val:   {val_s3_path}")

In [ ]:
# Clean up local files
import shutil
shutil.rmtree("./sft_data")
print("Local files cleaned up.")

## 5. Register Dataset in SageMaker AI Registry

The SageMaker AI Registry provides a centralized catalog for datasets, models, and evaluators. Registering the dataset makes it discoverable and reusable across training jobs.

We use `DataSet.create()` with `CustomizationTechnique.SFT` to register the training and validation datasets.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

dataset_train = DataSet.create(
    name="medical-triage-sft-train",
    source=train_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)
print(f"Training dataset ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="medical-triage-sft-val",
    source=val_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)
print(f"Validation dataset ARN: {dataset_val.arn}")

### Log Dataset to MLflow

We also log the dataset metadata to the MLflow experiment for full lineage tracking.

In [ ]:
with mlflow.start_run(run_name="dataset-curation"):
    mlflow.log_param("source_experiments", f"{agent_experiment_name}, {datacapture_experiment_name}, {offline_eval_experiment_name}")
    mlflow.log_param("total_traces", len(all_traces))
    mlflow.log_param("curated_from_traces", len(curated_records))
    mlflow.log_param("train_samples", len(train_records))
    mlflow.log_param("val_samples", len(val_records))
    mlflow.log_param("train_s3_path", train_s3_path)
    mlflow.log_param("val_s3_path", val_s3_path)
    mlflow.log_param("train_dataset_arn", dataset_train.arn)
    mlflow.log_param("val_dataset_arn", dataset_val.arn)
    mlflow.set_tag("stage", "dataset-curation")
    mlflow.set_tag("teacher_model", "qwen3-8b")

print("Dataset metadata logged to MLflow.")

## 6. Store Variables for Next Notebook

We store the key variables so the fine-tuning notebook can pick them up.

In [ ]:
%store train_s3_path
%store val_s3_path
%store ft_experiment_name

print("Variables stored for fine-tuning notebook:")
print(f"  train_s3_path = {train_s3_path}")
print(f"  val_s3_path = {val_s3_path}")
print(f"  ft_experiment_name = {ft_experiment_name}")

## Next Steps

You have curated a high-quality training dataset from production traces across multiple MLflow experiments and registered it in the SageMaker AI Registry.

In the next notebook, you will use this dataset to fine-tune a smaller Qwen2.5-7B model via Supervised Fine-Tuning (SFT), deploy it, and test it against the same medical triage queries.